In [6]:
import numpy as np
import pandas as pd 
import json
from hbv_bmi_project import HBV_Bmi
import xarray as xr

In [3]:
df = pd.read_csv("ERA5_Temp_1980_2026.csv")

# Parse year and month from the index (format: YYYYMM)
df['year'] = df['system:index'].astype(str).str[:4].astype(int)
df['month'] = df['system:index'].astype(str).str[4:6].astype(int)
 
# Convert temperature from Kelvin to Celsius
df['temperature_C'] = df['temperature_2m'] - 273.15
 
# Calculate mean annual temperature (average all months per year)
annual_mean = df.groupby('year')['temperature_C'].mean()
 
# Assign each year to a decade (e.g., 1980-1989 → 1980s)
annual_mean_df = annual_mean.reset_index()
annual_mean_df.columns = ['year', 'mean_annual_temp_C']
annual_mean_df['decade'] = (annual_mean_df['year'] // 10) * 10
 
# Calculate mean temperature per decade
decadal_mean = annual_mean_df.groupby('decade')['mean_annual_temp_C'].mean().reset_index()
decadal_mean.columns = ['decade', 'mean_decadal_temp_C']
 
print("Mean Annual Temperature per Year:")
print(annual_mean_df.to_string(index=False))
print("\nMean Temperature per Decade:")
print(decadal_mean.to_string(index=False))
 
# Pivot table: rows = years, columns = months (Jan=1 ... Dec=12)
pivot = df.pivot_table(index='year', columns='month', values='temperature_C')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot.index.name = 'year'
 
print("\nMonthly Mean Temperature per Year (°C):")
print(pivot.round(2).to_string())

Mean Annual Temperature per Year:
 year  mean_annual_temp_C  decade
 1980            7.973034    1980
 1981            8.474816    1980
 1982            9.249956    1980
 1983            9.250297    1980
 1984            8.350761    1980
 1985            7.820802    1980
 1986            8.149693    1980
 1987            8.072648    1980
 1988            9.459149    1980
 1989            9.858341    1980
 1990            9.866679    1990
 1991            8.950346    1990
 1992            9.638510    1990
 1993            9.042281    1990
 1994           10.184077    1990
 1995            9.518560    1990
 1996            7.946916    1990
 1997            9.287314    1990
 1998            9.076644    1990
 1999            9.614176    1990
 2000            9.938774    2000
 2001            9.376367    2000
 2002            9.934531    2000
 2003           10.092163    2000
 2004            9.192590    2000
 2005            9.413167    2000
 2006            9.978638    2000
 2007         

In [ ]:
grdc_obs = pd.read_csv(
    "/home/group3/teaching-materials/book/2_modelling_advanced_ewatercycle/Forcing/6336500_Q_Day.Cmd.txt",
    encoding="latin-1",
    sep=";",
    skiprows=36,
    index_col=0,
    parse_dates=True,
    usecols=[0, 2],
)
grdc_obs.index.name = "date"
grdc_obs.columns = ["Observations from GRDC"]

shape_area = 28191724718

In [ ]:
#                   Imax Ce Sumax beta Pmax   Tlag   Kf  Ks Gamma
ParMinn = np.array([0,   0.2,  40,    .5,   .001,   0,     .01,  .0001, .0001])
ParMaxn = np.array([8,    1,  800,   4,    .3,     10,    .1,   .01, 1])
Sin = np.array([0,  100,  0,  5  ])

In [ ]:
#import and clean data
# fix the initialization with T_baseline = 8.99 
# need Q_obs

In [ ]:
def nse(obs, sim):
    """Nash-Sutcliffe Efficiency"""
    return 1 - (np.sum((obs - sim)**2) / np.sum((obs - np.mean(obs))**2))

N = 100 # Number of runs
results = []

# Assuming you have observed discharge Q_obs as a numpy array or series
for i in range(N):
    # 1. Select random parameters from your ParMinn/ParMaxn
    p = np.random.uniform(ParMinn, ParMaxn)
    
    # 2. Setup and Initialize model
    model = HBV_Bmi()
    # You'll need a way to pass these params; 
    # either via the config file or using model.set_value('Sumax', p[2]) etc.
    model.initialize(config_file_path) 
    
    # 3. Run loop
    sim_q = []
    for _ in range(model.end_timestep):
        model.update()
        dest = np.array([0.0])
        model.get_value("Q", dest)
        sim_q.append(dest[0])
    
    # 4. Evaluate
    score = nse(Q_obs, np.array(sim_q))
    results.append((score, p))

# Find the best parameter set
best_run = max(results, key=lambda x: x[0])
print(f"Best NSE: {best_run[0]}")
print(f"Best Parameters: {best_run[1]}")